# Run the checkpoint after each training.

In [1]:
checkpoint_path = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251113_231525/checkpoint_epoch_0000_20251114_025531.pkl"

In [2]:
import torch
import pickle
import os
import sys

def extract_nested_state_dict(original_path):
    """
    从嵌套结构中提取模型状态字典
    """
    print(f"🔍 提取嵌套结构中的模型状态字典...")
    
    try:
        # 加载原始数据
        with open(original_path, 'rb') as f:
            checkpoint_data = pickle.load(f)
        
        print("✅ 成功加载检查点")
        
        # 深入嵌套结构
        if 'model' in checkpoint_data and isinstance(checkpoint_data['model'], dict):
            model_data = checkpoint_data['model']
            print(f"找到 'model' 字典，包含键: {list(model_data.keys())}")
            
            # 在 model 字典中查找 model_state_dict
            if 'model_state_dict' in model_data:
                model_state_dict = model_data['model_state_dict']
                if isinstance(model_state_dict, dict):
                    tensor_count = sum(1 for v in model_state_dict.values() if hasattr(v, 'shape'))
                    print(f"✅ 找到 model_state_dict，包含 {tensor_count} 个张量")
                    return model_state_dict
                else:
                    print(f"❌ model_state_dict 不是字典，而是 {type(model_state_dict)}")
            else:
                print("❌ 在 model 字典中未找到 model_state_dict")
        
        print("❌ 无法找到模型状态字典")
        return None
        
    except Exception as e:
        print(f"❌ 提取失败: {e}")
        return None

def create_fixed_checkpoint(original_path):
    """
    创建修复后的检查点文件
    """
    print(f"\n💾 创建修复后的检查点文件...")
    
    try:
        # 提取模型状态字典
        model_state_dict = extract_nested_state_dict(original_path)
        
        if model_state_dict is None:
            print("❌ 无法提取模型状态字典，无法修复")
            return None
        
        # 加载原始数据以保留元数据
        with open(original_path, 'rb') as f:
            original_data = pickle.load(f)
        
        # 创建标准检查点结构
        standard_checkpoint = {
            'model_state_dict': model_state_dict,
            'epoch': original_data.get('epoch', 0),
            'timestamp': original_data.get('timestamp', ''),
            'metrics': original_data.get('metrics', {}),
            'config': original_data.get('additional_info', {}),
            'optimizer_state_dict': original_data.get('model', {}).get('optimizer_state_dict'),
            'scheduler_state_dict': original_data.get('model', {}).get('scheduler_state_dict'),
            'experiment_state': original_data.get('model', {}).get('experiment_state'),
            'best_metric': original_data.get('model', {}).get('best_metric'),
            'best_epoch': original_data.get('model', {}).get('best_epoch'),
            'framework': 'pytorch'
        }
        
        # 保存为标准格式
        repaired_path = original_path.replace('.pkl', '_fixed.pth')
        torch.save(standard_checkpoint, repaired_path)
        print(f"✅ 修复后的检查点已保存: {repaired_path}")
        
        # 验证新文件
        print(f"\n🔍 验证修复后的文件...")
        verified = torch.load(repaired_path, map_location='cpu')
        print(f"✅ 验证成功! 修复后文件包含:")
        for key, value in verified.items():
            if key == 'model_state_dict':
                print(f"  {key}: {len(value)} 个参数")
            elif isinstance(value, dict):
                print(f"  {key}: 字典 ({len(value)} 个键)")
            else:
                print(f"  {key}: {type(value)}")
        
        return repaired_path
        
    except Exception as e:
        print(f"❌ 创建修复检查点失败: {e}")
        return None

def analyze_model_parameters(model_state_dict):
    """
    分析模型参数结构
    """
    print(f"\n📊 分析模型参数结构...")
    
    if model_state_dict is None:
        print("❌ 没有模型状态字典可供分析")
        return
    
    print(f"总参数数量: {len(model_state_dict)}")
    
    # 按模块分组参数
    modules = {}
    for key in model_state_dict.keys():
        parts = key.split('.')
        if len(parts) >= 2:
            module_name = parts[0] + '.' + parts[1]  # 例如 'chaotic_embedding.initial_state_mapper'
        else:
            module_name = key
        
        if module_name not in modules:
            modules[module_name] = []
        modules[module_name].append((key, model_state_dict[key].shape))
    
    print("参数按模块分组:")
    for module, params in modules.items():
        print(f"  {module}: {len(params)} 个参数")
        for key, shape in params[:2]:  # 每个模块只显示前2个参数
            print(f"    {key}: {shape}")
    
    # 特别关注分类器
    classifier_params = [(k, v.shape) for k, v in model_state_dict.items() if 'classifier' in k]
    if classifier_params:
        print(f"\n🎯 分类器参数:")
        for key, shape in classifier_params:
            print(f"  {key}: {shape}")
            if 'weight' in key and len(shape) == 2:
                print(f"    → 这个模型设计用于 {shape[0]} 个speakers")
    
    # 显示参数统计
    total_elements = sum(v.numel() for v in model_state_dict.values() if hasattr(v, 'numel'))
    print(f"\n📈 参数统计:")
    print(f"  总参数量: {total_elements:,}")
    print(f"  模型大小: {total_elements * 4 / (1024*1024):.2f} MB (假设float32)")

def create_loading_solution(original_path, repaired_path):
    """
    创建加载解决方案
    """
    print(f"\n🔧 创建加载解决方案...")
    
    # 方案1: 使用修复后的文件
    print("方案1: 使用修复后的文件 (推荐)")
    print(f"```python")
    print(f"checkpoint = torch.load('{repaired_path}', map_location='cpu')")
    print(f"model.load_state_dict(checkpoint['model_state_dict'])")
    print(f"```")
    
    # 方案2: 直接加载原始文件的代码
    print(f"\n方案2: 直接加载原始文件")
    print(f"```python")
    print(f"import pickle")
    print(f"")
    print(f"with open('{original_path}', 'rb') as f:")
    print(f"    data = pickle.load(f)")
    print(f"")
    print(f"# 提取嵌套的模型状态字典")
    print(f"model_state_dict = data['model']['model_state_dict']")
    print(f"model.load_state_dict(model_state_dict)")
    print(f"```")
    
    # 方案3: 通用加载函数
    print(f"\n方案3: 通用加载函数")
    print(f"```python")
    print(f"def load_any_checkpoint(checkpoint_path):")
    print(f"    \"\"\"加载任何格式的检查点\"\"\"")
    print(f"    try:")
    print(f"        # 首先尝试标准PyTorch加载")
    print(f"        checkpoint = torch.load(checkpoint_path, map_location='cpu')")
    print(f"        if 'model_state_dict' in checkpoint:")
    print(f"            return checkpoint['model_state_dict']")
    print(f"        else:")
    print(f"            return checkpoint")
    print(f"    except:")
    print(f"        # 如果失败，使用pickle加载")
    print(f"        with open(checkpoint_path, 'rb') as f:")
    print(f"            data = pickle.load(f)")
    print(f"        ")
    print(f"        # 处理嵌套结构")
    print(f"        if 'model' in data and 'model_state_dict' in data['model']:")
    print(f"            return data['model']['model_state_dict']")
    print(f"        elif 'model_state_dict' in data:")
    print(f"            return data['model_state_dict']")
    print(f"        elif 'model' in data:")
    print(f"            return data['model']")
    print(f"        else:")
    print(f"            return data")
    print(f"")
    print(f"# 使用")
    print(f"model_state_dict = load_any_checkpoint('{original_path}')")
    print(f"model.load_state_dict(model_state_dict)")
    print(f"```")

def test_parameter_loading(original_path, model_class, model_kwargs):
    """
    测试参数加载
    """
    print(f"\n🧪 测试参数加载...")
    
    try:
        # 提取模型状态字典
        model_state_dict = extract_nested_state_dict(original_path)
        
        if model_state_dict is None:
            print("❌ 无法提取模型状态字典，无法测试")
            return
        
        # 从状态字典推断模型配置
        if 'classifier.weight' in model_state_dict:
            num_speakers = model_state_dict['classifier.weight'].shape[0]
            print(f"从检查点推断: num_speakers = {num_speakers}")
            model_kwargs['num_speakers'] = num_speakers
        
        # 创建模型
        model = model_class(**model_kwargs)
        print(f"✅ 创建模型成功")
        
        # 尝试加载参数
        try:
            model.load_state_dict(model_state_dict, strict=True)
            print("✅ 严格模式加载成功!")
        except Exception as e:
            print(f"❌ 严格模式加载失败: {e}")
            print("尝试非严格模式...")
            model.load_state_dict(model_state_dict, strict=False)
            print("✅ 非严格模式加载成功")
        
        # 测试前向传播
        print(f"\n🔍 测试前向传播...")
        try:
            batch_size = 2
            audio_length = 16000
            test_input = torch.randn(batch_size, audio_length)
            
            with torch.no_grad():
                output = model(test_input)
                print(f"✅ 前向传播成功!")
                print(f"  输入: {test_input.shape}")
                print(f"  输出: {output.shape}")
        except Exception as e:
            print(f"❌ 前向传播失败: {e}")
        
    except Exception as e:
        print(f"❌ 测试失败: {e}")

# 主修复流程
if __name__ == "__main__":
    print("=" * 60)
    print("嵌套检查点修复工具")
    print("=" * 60)
    
    original_path = checkpoint_path
    
    # 1. 提取模型状态字典
    model_state_dict = extract_nested_state_dict(original_path)
    
    if model_state_dict:
        # 2. 分析模型参数
        analyze_model_parameters(model_state_dict)
        
        # 3. 创建修复后的检查点
        repaired_path = create_fixed_checkpoint(original_path)
        
        # 4. 创建加载解决方案
        create_loading_solution(original_path, repaired_path)
        
        # 5. 尝试测试参数加载（如果模型类可用）
        try:
            # 尝试导入模型类
            model_path = "/scratch/project_2003370/yueyao/Model"
            if model_path not in sys.path:
                sys.path.insert(0, model_path)
            
            from chaotic_network import ChaoticSpeakerRecognitionNetwork
            
            # 基本模型配置
            model_kwargs = {
                'embedding_dim': 10,
                'mlsa_scales': 5,
                'rqa_radius_ratio': 0.1,
                'evolution_time': 0.5,
                'time_step': 0.01,
                'coupling_strength': 1.0,
                'noise_level': 0.001,
                'pooling_type': 'comprehensive',
                'speaker_embedding_dim': 128,
                'classifier_type': 'cosine',
                'device': 'cpu'
            }
            
            test_parameter_loading(original_path, ChaoticSpeakerRecognitionNetwork, model_kwargs)
            
        except ImportError:
            print(f"\n⚠️ 无法导入模型类，跳过参数加载测试")
        
        print(f"\n🎉 修复完成!")
        if repaired_path:
            print(f"✅ 修复后的文件: {repaired_path}")
        print(f"💡 您现在可以使用上述任一方案加载检查点")
        
    else:
        print(f"\n❌ 无法提取模型状态字典")
        print(f"💡 检查点文件可能已损坏或不包含有效的模型参数")

嵌套检查点修复工具
🔍 提取嵌套结构中的模型状态字典...
✅ 成功加载检查点
找到 'model' 字典，包含键: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'experiment_state', 'config', 'best_metric', 'best_epoch']
✅ 找到 model_state_dict，包含 29 个张量

📊 分析模型参数结构...
总参数数量: 29
参数按模块分组:
  chaotic_embedding.initial_state_mapper: 4 个参数
    chaotic_embedding.initial_state_mapper.0.weight: torch.Size([16, 230])
    chaotic_embedding.initial_state_mapper.0.bias: torch.Size([16])
  chaotic_embedding.coupling_mapper: 4 个参数
    chaotic_embedding.coupling_mapper.0.weight: torch.Size([8, 230])
    chaotic_embedding.coupling_mapper.0.bias: torch.Size([8])
  chaotic_embedding.param_adapter: 4 个参数
    chaotic_embedding.param_adapter.0.weight: torch.Size([8, 230])
    chaotic_embedding.param_adapter.0.bias: torch.Size([8])
  speaker_embedding.embedding_network: 16 个参数
    speaker_embedding.embedding_network.0.weight: torch.Size([64, 5])
    speaker_embedding.embedding_network.0.bias: torch.Size([64])
  classifier.weight: 1 个

In [3]:
fixed_checkpoint_path = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251113_231525/checkpoint_epoch_0000_20251114_025531_fixed.pth"

# test_checkpoint

In [4]:
import torch

# 加载修复后的检查点
checkpoint = torch.load(fixed_checkpoint_path, map_location='cpu', weights_only=False)

print("检查点结构:")
for key, value in checkpoint.items():
    if key == 'model_state_dict':
        print(f"{key}: {len(value)} 个参数")
    else:
        print(f"{key}: {value}")

print(f"\n模型设计用于 {checkpoint['model_state_dict']['classifier.weight'].shape[0]} 个speakers")

检查点结构:
model_state_dict: 29 个参数
epoch: 0
timestamp: 2025-11-14T02:55:31.621558
metrics: {}
config: {}
optimizer_state_dict: {'state': {0: {'step': tensor(164.), 'exp_avg': tensor([[ 1.9940e-04, -1.5198e-04,  8.2463e-05,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [-4.1944e-04, -1.7687e-03,  9.7689e-05,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [-8.5330e-04, -1.3731e-03, -2.2566e-04,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        ...,
        [ 8.0686e-05,  1.0758e-04, -1.7335e-05,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 7.4685e-05,  7.3390e-05,  4.4045e-05,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 4.7531e-04,  1.5007e-03, -6.7016e-05,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00]]), 'exp_avg_sq': tensor([[1.4554e-06, 7.9954e-07, 2.0149e-07,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [1.2664e-05, 8.8020e-06, 2.1931e-06,  ..., 0.0000e+00, 0.0000e+00,
         

# check_epoch_info

In [5]:
import torch

checkpoint_path = fixed_checkpoint_path

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

print("=== 检查点详细信息 ===")
print(f"保存的 epoch: {checkpoint.get('epoch', '未找到')}")
print(f"最佳指标: {checkpoint.get('best_metric', '未找到')}")
print(f"最佳 epoch: {checkpoint.get('best_epoch', '未找到')}")
print(f"时间戳: {checkpoint.get('timestamp', '未找到')}")

# 检查实验状态
if 'experiment_state' in checkpoint:
    exp_state = checkpoint['experiment_state']
    print(f"实验状态中的 epoch: {exp_state.get('epoch', '未找到')}")

=== 检查点详细信息 ===
保存的 epoch: 0
最佳指标: 0.02691924199461937
最佳 epoch: 31
时间戳: 2025-11-14T02:55:31.621558
实验状态中的 epoch: 31


# fix_checkpoint_epoch

In [6]:
import torch

checkpoint_path = fixed_checkpoint_path
# 加载检查点
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

print("修正前:")
print(f"保存的 epoch: {checkpoint.get('epoch')}")
print(f"最佳 epoch: {checkpoint.get('best_epoch')}")
print(f"实验状态中的 epoch: {checkpoint.get('experiment_state', {}).get('epoch')}")

# 修正epoch信息 - 使用实验状态中的epoch
correct_epoch = checkpoint.get('experiment_state', {}).get('epoch', 5)
checkpoint['epoch'] = correct_epoch

print(f"\n修正后:")
print(f"保存的 epoch: {checkpoint.get('epoch')}")

# 保存修正后的检查点
fixed_path = checkpoint_path.replace('.pth', '_corrected.pth')
torch.save(checkpoint, fixed_path)
print(f"修正后的检查点保存到: {fixed_path}")

修正前:
保存的 epoch: 0
最佳 epoch: 31
实验状态中的 epoch: 31

修正后:
保存的 epoch: 31
修正后的检查点保存到: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251113_231525/checkpoint_epoch_0000_20251114_025531_fixed_corrected.pth


In [7]:
!python comprehensive_diagnostic.py

🤖 全自动模型诊断系统
🔍 开始全面模型诊断...
📁 检查项目结构...
  ✅ 训练脚本: ['train_chaotic.py']
  ✅ 配置文件: ['outputs/baselines/experiments/baseline_mel_mlp_run_0/config.json', 'outputs/baselines/experiments/baseline_mfcc_mlp_run_0/config.json', 'outputs/baselines/experiments/baseline_mel_cnn_run_0/config.json', 'outputs/baselines/experiments/baseline_mfcc_cnn_run_0/config.json', 'outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_1/config.json', 'outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/config.json', 'outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_2/config.json', 'outputs/chaotic/experiments/chaotic_chua_full_chaotic_run_0/config.json', 'outputs/chaotic/experiments/chaotic_rossler_full_chaotic_run_2/config.json', 'outputs/chaotic/experiments/chaotic_rossler_full_chaotic_run_0/config.json', 'outputs/chaotic/experiments/chaotic_rossler_full_chaotic_run_1/config.json']
⚙️ 分析配置参数...
  📋 分析配置文件: outputs/baselines/experiments/baseline_mel_mlp_run_0/config.json
  📋 分析配置文件: outp

In [8]:
!python simplified_test.py

🚀 开始简化模型测试
🧪 开始基础学习能力测试...
📊 测试配置:
  说话人数量: 10
  特征维度: 230
  模型参数: 38,474
  数据样本: 1000
  Epoch  0: Loss = 2.3360, Acc = 0.1250
  Epoch  5: Loss = 1.3678, Acc = 0.8770
  Epoch 10: Loss = 0.5947, Acc = 0.9950
  Epoch 15: Loss = 0.1765, Acc = 0.9990

📈 测试结果:
  最终准确率: 0.9990
  最终损失: 0.0592
✅ 测试通过: 模型具备基本学习能力

🎵 分析特征提取质量...
  特征分析: 需要实现具体检查逻辑

🎉 基础测试通过! 问题可能在于:
   - 模型容量不足
   - 特征提取不适合真实数据
   - 超参数需要调整

💡 建议运行完整诊断获取详细分析
